# P04 — UK House Price Dynamics
## Notebook 01: Data Profiling and Preparation

This notebook loads, profiles and prepares four datasets for the P04 spatial
affordability analysis:

1. HM Land Registry Price Paid Data 2024 - individual property transactions
2. ONS House Price Statistics for Small Areas - LA-level annual medians
3. ONS Postcode Directory - resolves PPD postcodes to local authority codes
4. ONS Annual Survey of Hours and Earnings - median gross earnings by LA

By the end of this notebook we have a single analysis-ready file:
house_prices_final.csv - one row per English local authority, containing
median house price, median earnings, affordability ratio and region.

In [18]:
import os
os.getcwd()

'C:\\Users\\TEST\\OneDrive\\Documents\\The United Kingdom\\Jobs\\Data Science\\portfolio\\project4\\notebooks'

## Section 1 — Environment Setup

We import all libraries, add the project root to the Python path so that
config.py is importable from the notebooks/ subfolder, and create all output
directories if they do not already exist.

In [19]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import warnings

warnings.filterwarnings("ignore")

# config.py lives one level up in the project root
sys.path.insert(0, os.path.join(os.getcwd(), ".."))
import config

# Create all output directories if they do not already exist
for directory in [config.DATA_RAW, config.DATA_PROCESSED,
                  config.FIGURES_DIR, config.REPORTS_DIR, config.SRC_DIR]:
    os.makedirs(directory, exist_ok=True)

print("Python version:", sys.version)
print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)
print("\nDirectories confirmed:")
for directory in [config.DATA_RAW, config.DATA_PROCESSED,
                  config.FIGURES_DIR, config.REPORTS_DIR, config.SRC_DIR]:
    print(f"  {directory}")

Python version: 3.13.2 (tags/v3.13.2:4f8bb39, Feb  4 2025, 15:23:48) [MSC v.1942 64 bit (AMD64)]
Pandas version: 3.0.2
NumPy version: 2.4.4

Directories confirmed:
  C:\Users\TEST\OneDrive\Documents\The United Kingdom\Jobs\Data Science\portfolio\project4\data\raw
  C:\Users\TEST\OneDrive\Documents\The United Kingdom\Jobs\Data Science\portfolio\project4\data\processed
  C:\Users\TEST\OneDrive\Documents\The United Kingdom\Jobs\Data Science\portfolio\project4\figures
  C:\Users\TEST\OneDrive\Documents\The United Kingdom\Jobs\Data Science\portfolio\project4\reports
  C:\Users\TEST\OneDrive\Documents\The United Kingdom\Jobs\Data Science\portfolio\project4\src


## Section 2 — LAD to Region Lookup

We fetch the ONS ArcGIS LAD-to-region lookup using the same API endpoint
used in project02-nhs-regional-health-outcomes and project03-uk-crime-deprivation . This returns the 2024 local authority district codes (LAD24CD), local authority names (LAD24NM) and region names (RGN24NM) for all English local authorities.

This lookup is the geographic spine of the entire project. Every dataset we load will ultimately join to it on lad_code. We fetch it first so we can validate geographic coverage as each dataset is loaded.

In [20]:
response = requests.get(config.LAD_REGION_URL)
features = response.json()["features"]

lad_region = pd.DataFrame([f["attributes"] for f in features])
lad_region.columns = ["lad_code", "lad_name", "region_code", "region_name"]

print(f"LAD-region lookup: {lad_region.shape[0]} local authorities")
print(f"Regions: {sorted(lad_region['region_name'].unique())}")
print(f"\nMissing values:\n{lad_region.isna().sum()}")
lad_region.head()

LAD-region lookup: 296 local authorities
Regions: ['East Midlands', 'East of England', 'London', 'North East', 'North West', 'South East', 'South West', 'West Midlands', 'Yorkshire and The Humber']

Missing values:
lad_code       0
lad_name       0
region_code    0
region_name    0
dtype: int64


,lad_code,lad_name,region_code,region_name
0,E06000001,Hartlepool,E12000001,North East
1,E06000002,Middlesbrough,E12000001,North East
2,E06000003,Redcar and Cleveland,E12000001,North East
3,E06000004,Stockton-on-Tees,E12000001,North East
4,E06000005,Darlington,E12000001,North East


## Section 3 — Price Paid Data 2024

The HM Land Registry Price Paid Data 2024 contains every property sale in England and Wales registered with Land Registry during 2024. The file has
no column headers in the standard download format — we assign them from the published data specification.

The dataset contains two transaction categories. Category A covers standard residential sales — the market transactions we want to analyse. Category B
covers repossessions, buy-to-let transfers and other non-market transfers. We filter to Category A at load time.

The file is approximately 160 MB. We read it in chunks of 100,000 rows to avoid loading the full file into memory at once. Each chunk is filtered before being appended, so only Category A rows accumulate in memory.

Before running this cell, download the file from:
https://prod.publicdata.landregistry.gov.uk.s3-website-eu-west-1.amazonaws.com/pp-2024.csv
Save it to: data/raw/pp-2024.csv

In [21]:
chunks = []
total_rows = 0
filtered_rows = 0

for chunk in pd.read_csv(
    config.PPD_2024,
    header=None,
    names=config.PPD_COLUMNS,
    usecols=config.PPD_KEEP_COLUMNS,
    chunksize=config.PPD_CHUNK_SIZE,
    dtype=str,
    encoding="utf-8"
):
    total_rows += len(chunk)
    chunk = chunk[chunk["ppd_category"] == config.PPD_CATEGORY_A]
    filtered_rows += len(chunk)
    chunks.append(chunk)

ppd = pd.concat(chunks, ignore_index=True)

ppd["price"] = pd.to_numeric(ppd["price"], errors="coerce")
ppd["date_of_transfer"] = pd.to_datetime(
    ppd["date_of_transfer"], errors="coerce"
)

print(f"Total rows read:       {total_rows:,}")
print(f"Category A rows kept:  {filtered_rows:,}")
print(f"Final shape:           {ppd.shape}")
print(f"\nProperty types:\n{ppd['property_type'].value_counts()}")
print(f"\nPrice summary:\n{ppd['price'].describe().round(0)}")

Total rows read:       923,729
Category A rows kept:  758,965
Final shape:           (758965, 10)

Property types:
property_type
S    225892
T    203529
D    197864
F    131680
Name: count, dtype: int64

Price summary:
count      758965.0
mean       368846.0
std        398253.0
min             1.0
25%        195000.0
50%        290000.0
75%        430000.0
max      74933000.0
Name: price, dtype: float64


In [22]:
# One-time extraction — run once to create onspd_subset.csv from the full NSPL file
# After this cell has run successfully, it never needs to run again

NSPL_FULL_PATH = os.path.join(config.DATA_RAW, "ONSPD_full.csv")

print("Loading full NSPL file — this may take 60-90 seconds...")

nspl_full = pd.read_csv(
    NSPL_FULL_PATH,
    usecols=["pcds", "lad25cd"],
    dtype=str,
    encoding="utf-8"
)

print(f"Full NSPL loaded: {nspl_full.shape[0]:,} rows")

# Rename to our standard column names
nspl_full.columns = ["postcode", "lad_code"]

# Normalise postcode format
nspl_full["postcode"] = nspl_full["postcode"].str.strip().str.upper()

# Keep England only — LAD codes starting with E
onspd_england = nspl_full[nspl_full["lad_code"].str.startswith("E", na=False)].copy()

print(f"England postcodes retained: {onspd_england.shape[0]:,}")

# Save subset
onspd_england.to_csv(config.ONSPD_SUBSET, index=False)

print(f"Subset saved to: {config.ONSPD_SUBSET}")

Loading full NSPL file — this may take 60-90 seconds...
Full NSPL loaded: 2,723,596 rows
England postcodes retained: 2,267,522
Subset saved to: C:\Users\TEST\OneDrive\Documents\The United Kingdom\Jobs\Data Science\portfolio\project4\data\raw\onspd_subset.csv


## Section 4 — Postcode Normalisation and LAD Merge

Each transaction in the Price Paid Data is identified by postcode, not by
local authority code. To aggregate transactions to LA level we need to join
each postcode to its LAD code using the ONS Postcode Directory (ONSPD).

Before joining we normalise both postcode fields to the same format:
stripped of whitespace and uppercased. A single trailing space or lowercase
character would cause a postcode to fail the join silently, producing a
missing LAD code with no error message.

We also drop ppd_category at this point -- it has served its purpose as a
filter and is no longer needed in the analysis dataset.

In [23]:
# Drop ppd_category if still present — may already be absent if kernel was restarted
ppd = ppd.drop(columns=["ppd_category"], errors="ignore")

# Normalise postcode
ppd["postcode"] = ppd["postcode"].str.strip().str.upper()

# Load ONSPD subset
onspd = pd.read_csv(config.ONSPD_SUBSET, dtype=str)
onspd["postcode"] = onspd["postcode"].str.strip().str.upper()

print(f"ONSPD subset loaded: {onspd.shape[0]:,} postcodes")

# Merge
ppd_geo = ppd.merge(onspd, on="postcode", how="left")

matched   = ppd_geo["lad_code"].notna().sum()
unmatched = ppd_geo["lad_code"].isna().sum()

print(f"\nMatched to LAD:   {matched:,}  ({matched/len(ppd_geo)*100:.1f}%)")
print(f"Unmatched:        {unmatched:,}  ({unmatched/len(ppd_geo)*100:.1f}%)")

# Keep England only and drop unmatched
ppd_geo = ppd_geo[ppd_geo["lad_code"].str.startswith("E", na=False)].copy()

print(f"\nEnglish transactions retained: {ppd_geo.shape[0]:,}")
print(f"Property types:\n{ppd_geo['property_type'].value_counts()}")

ONSPD subset loaded: 2,267,522 postcodes

Matched to LAD:   720,215  (94.9%)
Unmatched:        38,750  (5.1%)

English transactions retained: 720,215
Property types:
property_type
S    214014
T    190953
D    186486
F    128762
Name: count, dtype: int64


## Section 5 — ONS House Price Statistics for Small Areas

The ONS House Price Statistics for Small Areas dataset provides annual
median, mean, lower quartile and 10th percentile house prices by local
authority for England and Wales, derived from HM Land Registry Price Paid
Data and quality-assured by ONS.

We use this dataset in two ways. First, as a cross-validation check against
our own PPD aggregation — if our median prices match the ONS medians closely
we have confidence our aggregation is correct. Second, as the source of the
historical price series used in notebook 02 to show post-COVID price growth
by region.

Download instructions:
- Go to: https://www.ons.gov.uk/datasets/house-prices-local-authority
- Select Year ending December 2024
- Download the CSV and save as: data/raw/hpssa_la.csv

In [25]:
hpssa = pd.read_csv(config.HPSSA_LA, dtype=str)

print(f"HPSSA shape: {hpssa.shape}")
print(f"\nColumns:\n{hpssa.columns.tolist()}")
print(f"\nFirst few rows:")
hpssa.head()

HPSSA shape: (741600, 14)

Columns:
['V4_1', 'Data Marking', 'calendar-years', 'Time', 'mmm', 'Month', 'administrative-geography', 'Geography', 'property-type', 'PropertyType', 'build-status', 'BuildStatus', 'house-sales-and-prices', 'HouseSalesAndPrices']

First few rows:


,V4_1,Data Marking,calendar-years,Time,mmm,Month,administrative-geography,Geography,property-type,PropertyType,build-status,BuildStatus,house-sales-and-prices,HouseSalesAndPrices
0,375.0,NaN,2019,2019,mar,March,E06000001,Hartlepool,detached,Detached,all,All,sales,Count of sales
1,370.0,NaN,2020,2020,dec,December,E06000001,Hartlepool,detached,Detached,all,All,sales,Count of sales
2,391.0,NaN,2020,2020,mar,March,E06000001,Hartlepool,terraced,Terraced,all,All,sales,Count of sales
3,327.0,NaN,2020,2020,sep,September,E06000001,Hartlepool,terraced,Terraced,all,All,sales,Count of sales
4,57.0,NaN,2018,2018,dec,December,E06000001,Hartlepool,semi-detached,Semi-detached,newly-built,Newly built,sales,Count of sales


In [26]:
print("Unique measures:")
print(hpssa["house-sales-and-prices"].unique())

print("\nUnique property types:")
print(hpssa["property-type"].unique())

print("\nUnique build status:")
print(hpssa["build-status"].unique())

print("\nUnique years:")
print(sorted(hpssa["calendar-years"].unique()))

print("\nNull V4_1 count:", hpssa["V4_1"].isna().sum())
print("Data Marking values:", hpssa["Data Marking"].unique())

Unique measures:
<StringArray>
['sales', 'median', 'mean', 'lower-quartile', 'tenth-percentile']
Length: 5, dtype: str

Unique property types:
<StringArray>
['detached', 'terraced', 'semi-detached', 'all', 'flat-maisonette']
Length: 5, dtype: str

Unique build status:
<StringArray>
['all', 'newly-built', 'existing']
Length: 3, dtype: str

Unique years:
['2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022']

Null V4_1 count: 90949
Data Marking values: <StringArray>
[nan, '.']
Length: 2, dtype: str


In [27]:
# Filter to median price, all property types, all build status
hpssa_median = hpssa[
    (hpssa["house-sales-and-prices"] == "median") &
    (hpssa["property-type"] == "all") &
    (hpssa["build-status"] == "all")
].copy()

# Drop suppressed cells
hpssa_median = hpssa_median[hpssa_median["Data Marking"].isna()].copy()

# Keep and rename columns we need
hpssa_median = hpssa_median[[
    "administrative-geography", "Geography", "calendar-years", "V4_1"
]].copy()

hpssa_median.columns = ["lad_code", "lad_name", "year", "median_price_ons"]

# Convert median price to numeric
hpssa_median["median_price_ons"] = pd.to_numeric(
    hpssa_median["median_price_ons"], errors="coerce"
)

# Keep England only
hpssa_median = hpssa_median[
    hpssa_median["lad_code"].str.startswith("E", na=False)
].copy()

print(f"HPSSA median rows: {hpssa_median.shape[0]:,}")
print(f"Years covered: {sorted(hpssa_median['year'].unique())}")
print(f"LAs covered: {hpssa_median['lad_code'].nunique()}")
print(f"\nMedian price summary:")
print(hpssa_median["median_price_ons"].describe().round(0))
hpssa_median.head()

HPSSA median rows: 8,961
Years covered: ['2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022']
LAs covered: 309

Median price summary:
count       8961.0
mean      270102.0
std       141211.0
min        77500.0
25%       171000.0
50%       237000.0
75%       326000.0
max      1450000.0
Name: median_price_ons, dtype: float64


,lad_code,lad_name,year,median_price_ons
73,E06000001,Hartlepool,2015,124000.0
74,E06000001,Hartlepool,2018,124000.0
75,E06000001,Hartlepool,2020,128500.0
128,E06000001,Hartlepool,2016,119950.0
129,E06000001,Hartlepool,2019,126500.0


## Section 6 — ASHE Annual Gross Pay 2025

The Annual Survey of Hours and Earnings (ASHE) Table 7.7a provides median
gross annual pay by local authority on a place-of-work basis. We load the
All employees sheet directly from the Excel file, skipping the multi-row
header and extracting only the two columns we need: the LAD code and the
median gross annual pay.

Rows where the median is suppressed by ONS (marked x) are dropped. These
suppressions occur where the sample size is too small to produce a reliable
estimate. We keep only England local authorities identified by LAD codes
beginning with E.

The median rather than the mean is used throughout because earnings
distributions, like house price distributions, are right-skewed. The median
is ONS's own preferred measure for this reason, as stated in the ASHE
guidance notes.

In [29]:
import openpyxl

ASHE_PATH = os.path.join(config.DATA_RAW, "ashe_annual_gross_2025.xlsx")

# Row 5 is the header row (0-indexed: row 4), data starts at row 6 (0-indexed: row 5)
# We only need columns A (Description), B (Code), D (Median)
ashe_raw = pd.read_excel(
    ASHE_PATH,
    sheet_name="All",
    header=4,
    usecols=[0, 1, 3],
    dtype=str
)

# Assign clean column names
ashe_raw.columns = ["lad_name", "lad_code", "median_earnings"]

# Drop rows with no LAD code
ashe_raw = ashe_raw.dropna(subset=["lad_code"])

# Keep England LAD-level rows only (code starts with E0)
# E0 excludes regional aggregates (E12) and national aggregates (E92, K03 etc.)
ashe = ashe_raw[ashe_raw["lad_code"].str.startswith("E0", na=False)].copy()

# Replace suppressed values (x) with NaN and convert to numeric
ashe["median_earnings"] = ashe["median_earnings"].replace("x", np.nan)
ashe["median_earnings"] = pd.to_numeric(ashe["median_earnings"], errors="coerce")

# Drop suppressed rows
ashe = ashe.dropna(subset=["median_earnings"])

print(f"ASHE rows loaded:          {ashe_raw.shape[0]}")
print(f"England LA rows retained:  {ashe.shape[0]}")
print(f"Suppressed rows dropped:   {ashe_raw[ashe_raw['lad_code'].str.startswith('E0', na=False)].shape[0] - ashe.shape[0]}")
print(f"\nEarnings summary:")
print(ashe["median_earnings"].describe().round(0))
ashe.head()

ASHE rows loaded:          395
England LA rows retained:  295
Suppressed rows dropped:   1

Earnings summary:
count      295.0
mean     31799.0
std       5092.0
min      18675.0
25%      28662.0
50%      30636.0
75%      33543.0
max      67064.0
Name: median_earnings, dtype: float64


,lad_name,lad_code,median_earnings
5,Darlington UA,E06000005,29692.0
6,Hartlepool UA,E06000001,28204.0
7,Middlesbrough UA,E06000002,29502.0
8,Redcar and Cleveland UA,E06000003,28356.0
9,Stockton-on-Tees UA,E06000004,29537.0


## Section 7 — Aggregation, Merge and Export

We now aggregate the Price Paid Data from transaction level to local
authority level, computing the median price and transaction count per LA.
We then merge in four datasets on lad_code: the LAD-to-region lookup,
the ONS HPSSA median prices (cross-validation), and the ASHE earnings data.

The final step computes the affordability ratio for each LA:

    affordability_ratio = median_price_ppd / median_earnings

This ratio tells us how many years of median gross annual earnings it would
cost to buy the median-priced property in each local authority. The higher
the ratio, the less affordable the area.

We export the merged dataset as house_prices_final.csv — one row per
English local authority, containing all variables needed for notebook 02.

In [30]:
# Aggregate PPD to LA level
ppd_agg = (
    ppd_geo
    .groupby("lad_code")
    .agg(
        transaction_count=("price", "count"),
        median_price_ppd=("price", "median"),
        mean_price_ppd=("price", "mean"),
        lower_quartile_ppd=("price", lambda x: x.quantile(0.25))
    )
    .reset_index()
)

print(f"PPD aggregated to {ppd_agg.shape[0]} local authorities")
print(f"\nMedian price summary:")
print(ppd_agg["median_price_ppd"].describe().round(0))

PPD aggregated to 296 local authorities

Median price summary:
count        296.0
mean      332146.0
std       141809.0
min       130000.0
25%       231106.0
50%       305500.0
75%       400000.0
max      1235000.0
Name: median_price_ppd, dtype: float64


In [31]:
# Merge region lookup
df = ppd_agg.merge(lad_region, on="lad_code", how="left")

# Merge ONS HPSSA 2022 median for cross-validation
# Keep only 2022 (most recent year) for the cross-validation check
hpssa_2022 = hpssa_median[hpssa_median["year"] == "2022"][
    ["lad_code", "median_price_ons"]
].drop_duplicates("lad_code")

df = df.merge(hpssa_2022, on="lad_code", how="left")

# Merge ASHE earnings
df = df.merge(
    ashe[["lad_code", "median_earnings"]],
    on="lad_code",
    how="left"
)

# Compute affordability ratio
df["affordability_ratio"] = (
    df["median_price_ppd"] / df["median_earnings"]
)

print(f"Final merged shape: {df.shape}")
print(f"\nMissing values:")
print(df.isna().sum())
print(f"\nAffordability ratio summary:")
print(df["affordability_ratio"].describe().round(2))

Final merged shape: (296, 11)

Missing values:
lad_code               0
transaction_count      0
median_price_ppd       0
mean_price_ppd         0
lower_quartile_ppd     0
lad_name               2
region_code            2
region_name            2
median_price_ons       6
median_earnings        1
affordability_ratio    1
dtype: int64

Affordability ratio summary:
count    295.00
mean      10.34
std        3.67
min        4.57
25%        7.62
50%        9.94
75%       12.37
max       32.04
Name: affordability_ratio, dtype: float64


In [32]:
# Identify the two LAs missing region name
print("LAs missing region name:")
print(df[df["region_name"].isna()][["lad_code", "lad_name", "median_price_ppd", "transaction_count"]])

LAs missing region name:
      lad_code lad_name  median_price_ppd  transaction_count
261  E08000038      NaN          175000.0               3458
262  E08000039      NaN          213875.0               6378


In [33]:
# Manual fix for two South Yorkshire LAs that received new codes in 2025
# and are absent from the LAD24 ArcGIS lookup
boundary_fix = {
    "E08000038": ("Barnsley",          "E12000003", "Yorkshire and The Humber"),
    "E08000039": ("Sheffield",         "E12000003", "Yorkshire and The Humber"),
}

for lad_code, (name, region_code, region_name) in boundary_fix.items():
    mask = df["lad_code"] == lad_code
    df.loc[mask, "lad_name"]    = name
    df.loc[mask, "region_code"] = region_code
    df.loc[mask, "region_name"] = region_name

# Verify fix
print("Missing values after fix:")
print(df[["lad_name", "region_code", "region_name"]].isna().sum())
print(f"\nFixed rows:")
print(df[df["lad_code"].isin(["E08000038", "E08000039"])][
    ["lad_code", "lad_name", "region_name", "median_price_ppd"]
])

Missing values after fix:
lad_name       0
region_code    0
region_name    0
dtype: int64

Fixed rows:
      lad_code   lad_name               region_name  median_price_ppd
261  E08000038   Barnsley  Yorkshire and The Humber          175000.0
262  E08000039  Sheffield  Yorkshire and The Humber          213875.0


In [34]:
# Export final dataset
df.to_csv(config.HOUSE_PRICES_FINAL, index=False)

print(f"Exported: {config.HOUSE_PRICES_FINAL}")
print(f"Shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nFinal missing value check:")
print(df.isna().sum())
print(f"\nRegion distribution:")
print(df.groupby("region_name")["lad_code"].count().sort_values(ascending=False))

Exported: C:\Users\TEST\OneDrive\Documents\The United Kingdom\Jobs\Data Science\portfolio\project4\data\processed\house_prices_final.csv
Shape: (296, 11)

Columns: ['lad_code', 'transaction_count', 'median_price_ppd', 'mean_price_ppd', 'lower_quartile_ppd', 'lad_name', 'region_code', 'region_name', 'median_price_ons', 'median_earnings', 'affordability_ratio']

Final missing value check:
lad_code               0
transaction_count      0
median_price_ppd       0
mean_price_ppd         0
lower_quartile_ppd     0
lad_name               0
region_code            0
region_name            0
median_price_ons       6
median_earnings        1
affordability_ratio    1
dtype: int64

Region distribution:
region_name
South East                  64
East of England             45
East Midlands               35
North West                  35
London                      33
West Midlands               30
South West                  27
Yorkshire and The Humber    15
North East                  12
Name: lad

## Data Profiling Summary

| Dataset | Rows loaded | Rows retained | Notes |
|---|---|---|---|
| Price Paid Data 2024 | 923,729 | 720,215 | Category A, England only, postcode matched |
| NSPL postcode lookup | 2,267,522 | 2,267,522 | England only subset |
| ONS HPSSA | 741,600 | 8,961 | Median, all types, all build status, England |
| ASHE earnings | 395 | 295 | England LAs, valid medians only |
| house_prices_final.csv | 296 | 295 with affordability ratio | One row per English LA |